## Session 8, in class: profiling, and one equilibrium wrapper

Computational Macro (HWS 2026), University of Mannheim. The coding session after the midterm. Not graded.

Last week's notebook produced the same number two ways, by Monte Carlo simulation and by Young's method.
Part 1 of this notebook is that code again, given, run through once so that everything is defined. Part 2
measures: the two methods against each other, the three pieces of $A(r)$, and the profiler of the standard
library (the session 8 slides). Part 3 wraps the distribution into one general equilibrium solver that handles
Huggett and Aiyagari alike. Part 4 is yours: change the settings of the solver and watch what it costs.

**How to work with this notebook**

- Run the cells from top to bottom (`Shift + Enter`). After a restart of the kernel, start again at the top.
- There are **no gaps** this week. The code is complete; the work is to run it, read the output, and
  understand it: every timing of the slides is reproduced here on your machine.
- Part 1 has nothing to read: it is last week's code. Run it and move on.
- Each part ends with a **Check**: a number to compare with, from the slides.
- Part 4 has no check. It asks you to change the settings and read the table you get.

**Plan**

| Part | Content | Slides | Time |
| --- | --- | --- | --- |
| 1 | Last week's code: the household, the simulation, Young's method (given) | S6 | 5 min |
| 2 | Profiling: Monte Carlo against Young, the three pieces of $A(r)$, what takes so long | S8: 2 to 8 | 20 min |
| 3 | One equilibrium wrapper for Huggett and Aiyagari | S6: 18 to 22 | 15 min |
| 4 | Experiment: the settings of the solver, and what they cost | S8: 2, 3 | 20 min |


**Toolbox: the few tools this notebook uses that are new**

| Tool | What it does | Example |
| --- | --- | --- |
| `searchsortedlast(grid, x)` | the index `l` of the grid point just below (or at) `x` (for a sorted `grid`) | locating the lottery |
| `clamp(x, lo, hi)` | `x`, but not below `lo` and not above `hi` | keep `l` inside `1, ..., N_a - 1` |
| `reshape(1:n, na, nz)` | the numbers `1, ..., n` as an `na` by `nz` table | `idx[i, m]` is the row of state $(a_i, z_m)$ in `Q` |
| `sparse(rows, cols, vals, n, n)` | a sparse matrix from triplets: `vals[k]` sits at `(rows[k], cols[k])` | $\mathbf{Q}_{\mathbf{g}}$ with $2 N_z$ entries per row |
| `Q' * lam` | sparse matrix times vector | one step of the distribution |
| `rng = MersenneTwister(seed)`, `rand(rng, N)` | a seeded random number generator, `N` uniforms | the simulation |
| `@time f(x)`, `@elapsed f(x)` | time (and allocations) of one call, or the seconds as a number | part 2 |
| `@profile f(x)`, `Profile.print()` | the profiler of the standard library: where the time goes, by function | part 2 |

Everything else is from the earlier notebooks: `A * B`, the dots `.+`, `.*`, `interp1`, `sum(A, dims = 2)`.
A policy is stored as an $N_a \times N_z$ matrix: row `i` is the asset grid point $a_i$, column `m` the income state $z_m$.

### 0. Packages

Nothing new to install: `SparseArrays`, `Random`, `Statistics`, and `Profile` ship with Julia, and `QuantEcon`
(for the income chain) is in the course environment since week 5. The first cell activates the course
environment in `Julia/` and installs every package listed there that is missing on your laptop.

In [35]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))   # the course environment in Julia/
Pkg.instantiate()                        # installs every package it lists that is missing on your laptop

  Activating project at `c:\Users\manuc\OneDrive\Dokumente\GitHub\Manucher-Karimi-Computational_Macro-2026\Julia`


In [36]:
using Plots
using Printf
using LinearAlgebra
using SparseArrays                                        # sparse matrices, for the policy chain
using Random
using Statistics
using Profile                                             # the profiler, part 2
using QuantEcon                                           # rouwenhorst, as in week 5

### 1. Last week's code, given

The household of the session 6 slides, the Monte Carlo simulation, and Young's method, exactly as in
`Week_7/aggregation_and_young`, with the gaps filled. Nothing to read here that you have not seen, run the
cells and check that the last one prints mean assets $8.517$.

In [37]:
Base.@kwdef struct EconomicParameters
    γ::Float64 = 2.0          # relative risk aversion
    β::Float64 = 0.96         # discount factor
    ρ::Float64 = 0.9          # persistence of ln z
    sig_eps::Float64 = 0.2    # standard deviation of the innovation to ln z
end

Base.@kwdef struct NumericalParameters          # the numerical choices: grid sizes, tolerances
    na::Int = 200             # asset grid points N_a
    nz::Int = 7               # income grid points N_z
    amax::Float64 = 100.0     # upper grid bound a_{N_a}
    crit_pol::Float64 = 1e-8  # the EGM stops when the policy moves by less
    crit_dist::Float64 = 1e-12  # the distribution iteration stops when lam moves by less (part 3)
    maxiter::Int = 100_000    # iteration cap, the safety net
end

NumericalParameters

In [38]:
struct Grids                                    # a container for the grids, filled by make_grids
    a::Vector{Float64}        # asset grid, N_a
    z::Vector{Float64}        # income grid, N_z
    Pi::Matrix{Float64}       # transition matrix, N_z x N_z
    lam_z::Vector{Float64}    # stationary distribution of the income chain, N_z
end

function make_grids(amin, par, mpar)
    # The income chain of session 4 on N_z Rouwenhorst states (week 5, part 1), income levels z = exp(ln z),
    # and the quadratically spaced asset grid: a_i = amin + (amax - amin) s_i^2 with s_i equidistant on [0, 1]
    mc = rouwenhorst(mpar.nz, par.ρ, par.sig_eps)        # the chain for ln z, as in week 5
    z = exp.(mc.state_values)                            # income levels
    lam_z = vec(stationary_distributions(mc)[1])         # the stationary distribution of the chain, week 4
    s = collect(range(0.0, 1.0, length = mpar.na))       # N_a equidistant points on [0, 1]
    a = amin .+ (mpar.amax - amin) .* s .^ 2             # squared: dense near amin, sparse near amax
    return Grids(a, z, Matrix(mc.p), lam_z)
end

make_grids (generic function with 1 method)

In [39]:
par, mpar = EconomicParameters(), NumericalParameters()  # the default values listed above
gri = make_grids(0.0, par, mpar)                         # borrowing limit a >= 0 for parts 1 and 2
println("income states: ", round.(gri.z, digits = 3))
println("stationary distribution of income: ", round.(gri.lam_z, digits = 3), "  mean income E[z] = ", round(gri.lam_z' * gri.z, digits = 4))
println("asset grid: first spacing ", round(gri.a[2] - gri.a[1], digits = 4), "  last spacing ", round(gri.a[end] - gri.a[end-1], digits = 3))

income states: [0.325, 0.473, 0.688, 1.0, 1.454, 2.115, 3.077]
stationary distribution of income: [0.016, 0.094, 0.234, 0.312, 0.234, 0.094, 0.016]  mean income E[z] = 1.1103
asset grid: first spacing 0.0025  last spacing 1.002


In [40]:
# The dots inside make both functions work elementwise on a number, a vector, or a matrix
uprime(c, par) = c .^ (-par.γ)                  # marginal utility of CRRA utility u(c) = c^(1 - γ) / (1 - γ)
uprime_inv(m, par) = m .^ (-1 / par.γ)

uprime_inv (generic function with 1 method)

In [41]:
function interp1(xq, xgrid, fgrid)
    # Linear interpolation of the points (xgrid, fgrid) at every query point in xq, flat outside the grid (week 5)
    out = similar(xq)
    for (k, x) in enumerate(xq)
        i = clamp(searchsortedlast(xgrid, x), 1, length(xgrid) - 1)   # the interval [xgrid[i], xgrid[i+1]] around x
        w = clamp((x - xgrid[i]) / (xgrid[i+1] - xgrid[i]), 0.0, 1.0) # weight on the right end point, flat outside
        out[k] = (1 - w) * fgrid[i] + w * fgrid[i+1]
    end
    return out
end

interp1 (generic function with 1 method)

In [42]:
function egm_step(C, r, w, par, gri)
    # One step of the endogenous grid method (session 5, slide 20): from the policy C to the new policy on the grid
    R = 1 + r
    EMU = uprime(C, par) * gri.Pi'              # expected marginal utility tomorrow at every (a'_l, z_m), week 5 gap 5
    ctilde = uprime_inv(par.β * R .* EMU, par)  # (a) consumption today from the inverted Euler equation
    atilde = (ctilde .+ gri.a .- w .* gri.z') ./ R   # (b) today's assets from the budget constraint
    Cnew = similar(C)                           # (c) back to the grid, one income state at a time
    for m in 1:length(gri.z)
        Cnew[:, m] = interp1(gri.a, atilde[:, m], ctilde[:, m])
        below = gri.a .< atilde[1, m]           # poorer than the household that saves a' = a_1: the limit binds
        Cnew[below, m] = w * gri.z[m] .+ R .* gri.a[below] .- gri.a[1]   # consume everything down to a' = a_1
    end
    return Cnew
end

egm_step (generic function with 1 method)

In [43]:
function solve_household(r, w, par, mpar, gri; C0 = nothing)
    # The EGM loop of week 5: iterate egm_step until the policy stops moving. Returns C, the savings policy G, and the count.
    # C0: an optional starting guess for the policy (the solution at nearby prices, part 2)
    R = 1 + r
    C = C0 === nothing ? max.(r .* gri.a .+ w .* gri.z', 1e-6) : copy(C0)   # the guess: consume interest and income
    it = 0
    while it < mpar.maxiter
        it += 1
        Cnew = egm_step(C, r, w, par, gri)
        dist = maximum(abs.(Cnew .- C))         # largest change of the policy
        C = Cnew
        dist < mpar.crit_pol && break
    end
    G = w .* gri.z' .+ R .* gri.a .- C          # savings a' = w z + R a - c, N_a x N_z
    return C, G, it
end

solve_household (generic function with 1 method)

In [44]:
r0, w0 = 0.03, 1.0                              # the prices of parts 1 and 2
t0 = time()
C0, G0, it0 = solve_household(r0, w0, par, mpar, gri)
@printf("EGM: %d iterations, %.3f s\n", it0, time() - t0)
println("net saving g(a, z) - a at a = 0, by income state: ", round.(G0[1, :] .- gri.a[1], digits = 3))
i10 = argmin(abs.(gri.a .- 10))                 # the grid point closest to a = 10
println("net saving at a = ", round(gri.a[i10], digits = 2), ", by income state: ", round.(G0[i10, :] .- gri.a[i10], digits = 3))

EGM: 369 iterations, 1.069 s
net saving g(a, z) - a at a = 0, by income state: [0.0, 0.0, 0.005, 0.113, 0.368, 0.82, 1.547]
net saving at a = 10.02, by income state: [-0.566, -0.493, -0.369, -0.165, 0.157, 0.656, 1.415]


In [45]:
function simulate_panel(G, gri, N, T, seed)
    # N households for T periods. Returns the final cross section (assets, income index) and the mean of assets in every period
    rng = MersenneTwister(seed)                 # seeded, so that the run is reproducible
    cum = cumsum(gri.Pi, dims = 2)
    cum_z = cumsum(gri.lam_z)                   # cumulated stationary distribution, for the initial income
    a = fill(gri.a[1], N)                       # everybody starts at the limit
    m = [searchsortedfirst(cum_z, u) for u in rand(rng, N)]   # income indices drawn from the stationary distribution
    means = zeros(T)                            # mean assets in every period
    for t in 1:T
        anew = similar(a)
        for k in 1:length(gri.z)                # (i) tomorrow's assets: one interpolation per income state
            sel = m .== k                       # the households with income z_k today
            anew[sel] = interp1(a[sel], gri.a, G[:, k])
        end
        a = anew
        u = rand(rng, N)                        # (ii) tomorrow's income: one uniform per household
        m = [min(searchsortedfirst(view(cum, m[j], :), u[j]), length(gri.z)) for j in 1:N]   # the first n with F_mn >= u (week 5 by hand)
        means[t] = mean(a)
    end
    return a, m, means
end

simulate_panel (generic function with 1 method)

In [46]:
N, T = 10_000, 1_000
t0 = time()
a_sim, m_sim, means = simulate_panel(G0, gri, N, T, 11)
t_sim = time() - t0
@printf("simulation of N = %d households for T = %d periods: %.2f s\n", N, T, t_sim)
@printf("mean assets %.3f, share at the limit %.3f, median %.2f\n", mean(a_sim), mean(a_sim .<= gri.a[1] + 1e-10), median(a_sim))
println("mean assets over time, t = 10, 50, 100, 300, 1000: ", round.(means[[10, 50, 100, 300, 1000]], digits = 2))

simulation of N = 10000 households for T = 1000 periods: 6.46 s
mean assets 8.454, share at the limit 0.017, median 6.33
mean assets over time, t = 10, 50, 100, 300, 1000: [1.89, 5.64, 7.33, 8.56, 8.45]


In [47]:
mc_rows = []                                    # (N, mean over seeds, sd across seeds, seconds per simulation), for part 2
for Nn in (1_000, 10_000)
    ms = Float64[]
    tt = time()
    for seed in 0:2
        a_s, _, _ = simulate_panel(G0, gri, Nn, 500, seed)
        push!(ms, mean(a_s))
    end
    tt = (time() - tt) / 3
    push!(mc_rows, (Nn, mean(ms), std(ms, corrected = false), tt))
    @printf("N = %7d: mean assets over 3 seeds %.3f, sd across seeds %.3f, %.2f s per simulation\n", Nn, mean(ms), std(ms, corrected = false), tt)
end

N =    1000: mean assets over 3 seeds 8.453, sd across seeds 0.065, 0.31 s per simulation
N =   10000: mean assets over 3 seeds 8.545, sd across seeds 0.021, 3.14 s per simulation


In [48]:
function lottery(G, gri)
    # For every policy value a'_im: the index l of the grid point just below it, and the weight w_im on the point above
    na = length(gri.a)
    l = clamp.(searchsortedlast.(Ref(gri.a), G), 1, na - 1)   # locate: a_l <= a'_im < a_{l+1}, l at most N_a - 1
    w = (G .- gri.a[l]) ./ (gri.a[l .+ 1] .- gri.a[l])
    w = clamp.(w, 0.0, 1.0)                     # exactly at the top of the grid, w would be 1 + 1e-16
    return l, w
end

lottery (generic function with 1 method)

In [49]:
l0, w0_ = lottery(G0, gri)
println("weights inside [0, 1]: ", all(0 .<= w0_ .<= 1))
# The lottery reproduces the policy, except where the richest households at a = 100 would save beyond the grid
# (two points; the lottery caps them at a_N, and part 3 checks that no mass ever gets there)
println("lottery mean equals the policy (capped at a_N): ", isapprox((1 .- w0_) .* gri.a[l0] .+ w0_ .* gri.a[l0 .+ 1], min.(G0, gri.a[end])))
println("policy values above the top of the grid: ", sum(G0 .> gri.a[end]))
i17 = argmin(abs.(gri.a .- 17.0))               # the example of slide 12: a = 17.0, income z_6
@printf("a = %.2f, z_6: a' = %.2f lies between %.2f and %.2f, w = %.2f\n", gri.a[i17], G0[i17, 6], gri.a[l0[i17, 6]], gri.a[l0[i17, 6] + 1], w0_[i17, 6])

weights inside [0, 1]: true
lottery mean equals the policy (capped at a_N): true
policy values above the top of the grid: 2
a = 16.98, z_6: a' = 17.56 lies between 17.40 and 17.82, w = 0.40


In [50]:
# The row of Q for the household of the example, written out: destination, share, as on slide 13
i, m = i17, 6                                   # the cell (a_i, z_m) = (17.0, z_6)
l_im, w_im = l0[i, m], w0_[i, m]
rows = Tuple{Float64, Float64, Float64}[]
for n in 1:length(gri.z)                        # income tomorrow
    push!(rows, (gri.a[l_im], gri.z[n], gri.Pi[m, n] * (1 - w_im)))
    push!(rows, (gri.a[l_im + 1], gri.z[n], gri.Pi[m, n] * w_im))   # the share landing on (a_{l+1}, z_n)
end
@printf("mass leaving (a, z) = (%.2f, z_6), policy a' = %.2f, lottery between %.2f and %.2f with w = %.2f\n", gri.a[i], G0[i, m], gri.a[l_im], gri.a[l_im + 1], w_im)
println("   a' tomorrow   z tomorrow   share")
for (a_to, z_to, share) in rows
    share > 1e-4 && @printf("   %9.2f   %9.2f   %.4f\n", a_to, z_to, share)   # skip the tiny shares of far income states
end
@printf("shares add up to %.6f, mean of a' tomorrow %.4f (the policy says %.4f)\n", sum(s for (_, _, s) in rows), sum(a_to * s for (a_to, _, s) in rows), G0[i, m])

mass leaving (a, z) = (16.98, z_6), policy a' = 17.56, lottery between 17.40 and 17.82 with w = 0.40
   a' tomorrow   z tomorrow   share
       17.40        0.69   0.0006
       17.82        0.69   0.0004
       17.40        1.00   0.0123
       17.82        1.00   0.0081
       17.40        1.45   0.1170
       17.82        1.45   0.0775
       17.40        2.12   0.4485
       17.82        2.12   0.2968
       17.40        3.08   0.0233
       17.82        3.08   0.0154
shares add up to 1.000000, mean of a' tomorrow 17.5640 (the policy says 17.5640)


In [51]:
function transition_matrix(G, gri)
    # Q[(i, m), (l, n)] = Pi[m, n] (1 - w_im) and Q[(i, m), (l + 1, n)] = Pi[m, n] w_im: the policy chain as a sparse matrix
    na, nz = length(gri.a), length(gri.z)
    idx = reshape(1:(na * nz), na, nz)          # idx[i, m] = i + (m - 1) N_a: the row of state (a_i, z_m), the order of Howard's step (week 5)
    l, w = lottery(G, gri)                      # the lottery of every (i, m)
    rows, cols, vals = Int[], Int[], Float64[]  # the triplets, collected column block by column block
    for m in 1:nz                               # income today
        for n in 1:nz                           # income tomorrow
            # mass 1 - w_im goes to (a_l, z_n) ...
            append!(rows, idx[:, m]); append!(cols, idx[l[:, m], n]); append!(vals, gri.Pi[m, n] .* (1 .- w[:, m]))
            # ... and mass w_im to (a_{l+1}, z_n)
            append!(rows, idx[:, m]); append!(cols, idx[l[:, m] .+ 1, n]); append!(vals, gri.Pi[m, n] .* w[:, m])
        end
    end
    return sparse(rows, cols, vals, na * nz, na * nz)
end

transition_matrix (generic function with 1 method)

In [52]:
t0 = time()
Q0 = transition_matrix(G0, gri)
@printf("Q built in %.1f ms: size %s, %d stored entries, %.2f percent of the matrix\n", 1000 * (time() - t0), size(Q0), nnz(Q0), 100 * nnz(Q0) / size(Q0, 1)^2)
rowsums = vec(sum(Q0, dims = 2))
@printf("rows sum to one: largest deviation %.1e\n", maximum(abs.(rowsums .- 1)))
idx0 = reshape(1:(mpar.na * mpar.nz), mpar.na, mpar.nz)   # the same table, for looking up rows below
row = Vector(Q0[idx0[i17, 6], :])               # the row of the example cell, as a dense vector
@printf("row of (17.0, z_6): %d nonzero entries, largest %.4f, sum %.6f\n", count(!iszero, row), maximum(row), sum(row))

Q built in 411.0 ms: size (1400, 1400), 19600 stored entries, 1.00 percent of the matrix
rows sum to one: largest deviation 4.4e-16
row of (17.0, z_6): 14 nonzero entries, largest 0.4485, sum 1.000000


In [53]:
function stationary_dist(Q, mpar, gri; lam0 = nothing)
    # Iterate lam_{n+1} = Q' lam_n from the uniform distribution (or from lam0). Returns lam as an N_a x N_z table, and the count
    n = size(Q, 1)
    lam = lam0 === nothing ? fill(1 / n, n) : vec(lam0)   # the start, as a vector in the order of idx
    QT = sparse(Q')                             # the transpose once, stored as its own sparse matrix
    it = 0
    while it < mpar.maxiter
        it += 1
        lam_new = QT * lam
        dist = maximum(abs.(lam_new .- lam))    # largest change
        lam = lam_new
        dist < mpar.crit_dist && break
    end
    return reshape(lam, length(gri.a), length(gri.z)), it   # back to the (a, z) table, the same order as idx
end

stationary_dist (generic function with 1 method)

In [54]:
t0 = time()
lam0, itl = stationary_dist(Q0, mpar, gri)
t_young = time() - t0
@printf("Young's iteration: %d iterations to 1e-12, %.0f ms\n", itl, 1000 * t_young)
# One step by hand, for one cell: the mass arriving at the example cell (17.0, z_6) is the weighted column sum of slide 14
lam_vec = vec(lam0)                             # lam as a vector in the order of idx
col = Vector(Q0[:, idx0[i17, 6]])               # column (17.0, z_6) of Q: who sends mass there, and how much
@printf("mass arriving at (17.0, z_6): %.3e from %d cells, against lam there %.3e (equal: stationary)\n", col' * lam_vec, count(!iszero, col), lam0[i17, 6])

Young's iteration: 1267 iterations to 1e-12, 176 ms
mass arriving at (17.0, z_6): 1.244e-03 from 14 cells, against lam there 1.244e-03 (equal: stationary)


In [55]:
function mean_assets(lam, gri)
    # A = Σ_{i, m} lam_im a_i, with lam as an N_a x N_z table
    return sum(lam .* gri.a)
end

mean_assets (generic function with 1 method)

In [56]:
A0 = mean_assets(lam0, gri)
lam_a0 = vec(sum(lam0, dims = 2))               # the marginal distribution of assets
@printf("mean assets %.4f (simulation: %.4f)\n", A0, mean(a_sim))
@printf("share at the limit %.4f (simulation: %.4f)\n", sum(lam0[1, :]), mean(a_sim .<= gri.a[1] + 1e-10))
@printf("check 1, mass: %.12f\n", sum(lam0))
@printf("check 2, marginal over assets against lam_z: %.1e\n", maximum(abs.(vec(sum(lam0, dims = 1)) .- gri.lam_z)))
@printf("check 3, mass at the top of the grid: %.1e\n", sum(lam0[end, :]))
println("mean assets by income state: ", round.(vec(sum(lam0 .* gri.a, dims = 1)) ./ vec(sum(lam0, dims = 1)), digits = 1))
cdf0 = cumsum(lam_a0)
@printf("median %.1f, share below one year of mean income: %.2f\n", gri.a[searchsortedfirst(cdf0, 0.5)], cdf0[searchsortedlast(gri.a, gri.lam_z' * gri.z)])

mean assets 8.5173 (simulation: 8.4544)
share at the limit 0.0162 (simulation: 0.0171)
check 1, mass: 1.000000000000
check 2, marginal over assets against lam_z: 3.5e-14
check 3, mass at the top of the grid: 2.2e-09
mean assets by income state: [4.2, 5.1, 6.4, 8.1, 10.4, 13.4, 17.3]
median 6.3, share below one year of mean income: 0.14


### 2. Profiling: what takes so long?

*Session 8, slides 2 to 8.*

Last week's notebook produced the same number two ways. Before choosing, **measure**: guessing where the time goes is
the most common mistake in numerical work. Three levels, from cheap to thorough.

1. **A clock around a call.** `time.perf_counter()` before and after (Julia: `@time`, which also reports
   allocations). Repeat the call a few times and take the best: the first call of anything is slow (caches,
   and in Julia the compilation), and the machine does other things in the background.
2. **Clocks around the pieces.** $A(r)$ has three pieces: solve the household, build $\mathbf{Q}_{\mathbf{g}}$,
   iterate on $\boldsymbol{\lambda}$. Time them one by one and the picture is usually clear.
3. **A profiler.** It samples (Julia) or records (Python) which function is running and reports the time
   per function, with the calls inside counted separately. Use it when the piecewise clocks are not enough,
   or when the code is somebody else's.

**Monte Carlo against Young, in one table.** The first comparison is the one that matters: how much does one
digit of the mean cost?

In [57]:
function best_of(f; reps = 3)
    # the result of f() and the best of `reps` timings in seconds (the first call is slower: Julia compiles it)
    best, out = Inf, nothing
    for _ in 1:reps
        t0 = time()
        out = f()
        best = min(best, time() - t0)
    end
    return out, best
end

best_of (generic function with 1 method)

In [58]:
(lam_y, _), t_y = best_of(() -> stationary_dist(transition_matrix(G0, gri), mpar, gri))
@printf("%24s  %12s  %16s  %8s\n", "method", "mean assets", "sd across seeds", "seconds")
for (Nn, mean_mc, sd_mc, tt) in mc_rows
    @printf("%24s  %12.4f  %16.4f  %8.2f\n", "Monte Carlo, N = $Nn", mean_mc, sd_mc, tt)
end
@printf("%24s  %12.4f  %16.4f  %8.2f\n", "Young", mean_assets(lam_y, gri), 0.0, t_y)

                  method   mean assets   sd across seeds   seconds
   Monte Carlo, N = 1000        8.4533            0.0653      0.31
  Monte Carlo, N = 10000        8.5446            0.0208      3.14
                   Young        8.5173            0.0000      0.04


**Check:** Young's method is exact to the grid and takes about as long as the smallest simulation, whose mean
is off by a quarter in either direction. A simulation that matches Young to two digits needs $N = 10^5$ and a
hundred times the time. And Young's $A(r)$ is a smooth function of $r$, which part 3 needs.

**The three pieces of $A(r)$.** The same $A(r)$ that part 3 calls in every bisection step: solve the household,
build $\mathbf{Q}_{\mathbf{g}}$, find $\boldsymbol{\lambda}$. Which one is expensive?

In [59]:
(C_, G_, it_), t_hh = best_of(() -> solve_household(r0, w0, par, mpar, gri))
Q_, t_Q = best_of(() -> transition_matrix(G_, gri))
(lam_, it_lam), t_lam = best_of(() -> stationary_dist(Q_, mpar, gri))
@printf("solve the household:          %6.0f ms  (%d EGM iterations, %.2f ms each)\n", 1000 * t_hh, it_, 1000 * t_hh / it_)
@printf("build Q:                      %6.0f ms\n", 1000 * t_Q)
@printf("stationary distribution:      %6.0f ms  (%d iterations, %.3f ms each)\n", 1000 * t_lam, it_lam, 1000 * t_lam / it_lam)
@printf("total A(r):                   %6.0f ms\n", 1000 * (t_hh + t_Q + t_lam))

solve the household:              50 ms  (369 EGM iterations, 0.14 ms each)
build Q:                           0 ms
stationary distribution:          35 ms  (1267 iterations, 0.028 ms each)
total A(r):                       85 ms


**Check:** the household and the distribution each take a few hundred iterations of a cheap step, and
together they are the bill. Building $\mathbf{Q}_{\mathbf{g}}$ is a few milliseconds. (Julia: run the cell twice, the
first run includes compilation. `@time solve_household(r0, w0, par, mpar, gri)` also prints the allocations,
and a step that allocates a fresh array every iteration is the first thing to look at.)

**The profiler.** Python's `cProfile` records every function call and `pstats` sorts the result. Two columns
matter: `tottime` is the time spent inside the function itself, `cumtime` includes everything it calls. Sort by
`tottime` to find the hot spot, by `cumtime` to see the structure. Julia's `@profile` samples the call stack
many times per second and `Profile.print()` prints the tree, with the sample counts per line.

In [60]:
Profile.clear()
@profile begin
    C_, G_, _ = solve_household(r0, w0, par, mpar, gri)
    lam_, _ = stationary_dist(transition_matrix(G_, gri), mpar, gri)
end
Profile.print(format = :flat, sortedby = :count, mincount = 20)   # the lines with the most samples, flat

 Count  Overhead File                    Line Function
 =====  ======== ====                    ==== ========
    21         0 @Base\]8;;file:///C:/Users/manuc/.julia/juliaup/julia-1.12.7%2B0.x64.w64.mingw32/share/julia/base/broadcast.jl\broadcast.jl]8;;\      894 materialize
    21         0 @Base\]8;;file:///C:/Users/manuc/.julia/juliaup/julia-1.12.7%2B0.x64.w64.mingw32/share/julia/base/broadcast.jl\broadcast.jl]8;;\      919 copy
    25         0 In[43]                    1 solve_household(r::Float64, w::Fl…
    25         0 In[43]                    9 solve_household(r::Float64, w::Fl…
    42         0 @IJulia]8;;file:///C:/Users/manuc/.julia/packages/IJulia/Vl5w1/src/eventloop.jl\\src\eventloop.jl]8;;\   71 (::IJulia.var"#waitloop##2#waitlo…
    42         0 @IJulia]8;;file:///C:/Users/manuc/.julia/packages/IJulia/Vl5w1/src/eventloop.jl\\src\eventloop.jl]8;;\   26 eventloop(socket::ZMQ.Socket, ker…
    42         0 @IJulia\]8;;file:///C:/Users/manuc/.julia/package

**Reading the output.** In Python the top entries are `egm_step` itself (the elementwise work, $369$ calls),
the sparse matrix-vector product (called $1{,}267$ times), and `np.interp` inside `egm_step` ($7 \times 369$
calls). Nothing surprising, and nothing that a different line of code fixes: the heavy lifting already happens in
library calls. What is left is **fewer iterations**, and one way to fewer iterations is a better starting point.

**What that means for the code.** The profile confirms what the code already is: the heavy lifting happens
in library calls on whole arrays, the matrix is sparse, the solver is EGM. Nothing inside the code buys much.
Fewer iterations would, and part 3 uses the one cheap way to get them: inside the equilibrium loop, $A(r)$ is
evaluated at rates that get closer and closer, and the policy and the distribution at the previous $r$ are good
starting points for the next (`C0` and `lam0`). That saves about a quarter of the iterations, no more, because
both loops converge linearly: the count is $\ln(\text{start} / \text{tolerance}) / \ln(\text{rate})$, and a
start a hundred times closer removes two of eight orders of magnitude. The big factors were decided earlier, by
the algorithm: EGM against VFI ($0.04$ s against $3$), Young against Monte Carlo ($0.03$ against $7$).

### 3. One equilibrium wrapper for Huggett and Aiyagari 

*Session 6, slides 18 to 22.*

Both equilibria are one equation in one unknown, the interest rate: households supply assets, something
demands them, and the rate adjusts until supply equals demand. What differs is **who demands, and what the
wage is**:

| | Huggett (1993) | Aiyagari (1994) |
| --- | --- | --- |
| Asset | a bond, lent and borrowed between households | capital, rented to a firm |
| Borrowing limit $\underline{a}$ | $-2$ | $0$ |
| Wage $w(r)$ | $1$ | $(1 - \alpha) (K^d(r) / L)^\alpha$, from the firm |
| Demand $D(r)$ | $0$ (zero net supply) | $K^d(r) = L \bigl( (r + \delta) / \alpha \bigr)^{1 / (\alpha - 1)}$ |
| Equilibrium | $A(r^*) = 0$ | $A(r^*) = K^d(r^*)$ |

So one wrapper does both: an **economy** is a borrowing limit, a wage function, and a demand function. The
**distribution block** `aggregate` turns $(r, w)$ into $A(r)$ (household, $\mathbf{Q}_{\mathbf{g}}$,
$\boldsymbol{\lambda}$, sum), and `stationary_equilibrium` bisects on the excess supply $A(r) - D(r)$, which
increases in $r$, passing the warm start of part 2 from one step to the next. Keep the bracket below
$1 / \beta - 1$, where a patient household accumulates forever and $\boldsymbol{\lambda}$ does not exist.

In [61]:
struct Aggregates
    A::Float64                # mean assets A(r), the households' asset supply
    lam::Matrix{Float64}      # the stationary distribution, N_a x N_z
    C::Matrix{Float64}        # consumption policy
    G::Matrix{Float64}        # savings policy
end

function aggregate(r, w, gri; warm = nothing)
    # The distribution block: household -> policy chain -> stationary distribution -> mean assets, at the prices (r, w).
    # warm: the Aggregates of a nearby r, used as the starting point (part 2), or nothing for a cold start
    C, G, _ = solve_household(r, w, par, mpar, gri; C0 = warm === nothing ? nothing : warm.C)
    Q = transition_matrix(G, gri)
    lam, _ = stationary_dist(Q, mpar, gri; lam0 = warm === nothing ? nothing : warm.lam)
    return Aggregates(mean_assets(lam, gri), lam, C, G)
end

aggregate (generic function with 1 method)

In [62]:
struct Economy
    name::String
    gri::Grids                # the grids, with the borrowing limit of this economy
    wage::Function            # w(r)
    demand::Function          # D(r): what absorbs the households' assets at the rate r
end

function stationary_equilibrium(eco, lo, hi, tol)
    # Bisection on the excess supply A(r) - D(r), which increases in r, with warm starts along the way. Returns r*, the
    # aggregates at r*, and the step count
    warm, k = nothing, 0
    while hi - lo > tol
        mid = (lo + hi) / 2
        warm = aggregate(mid, eco.wage(mid), eco.gri; warm = warm)   # A(mid), started from the previous step's solution
        excess = warm.A - eco.demand(mid)
        # excess supply increases in r: if it is negative the root is above mid, keep [mid, hi], otherwise [lo, mid]
        lo, hi = excess < 0 ? (mid, hi) : (lo, mid)
        k += 1
    end
    return (lo + hi) / 2, warm, k
end

stationary_equilibrium (generic function with 1 method)

**Huggett (slides 18 and 19).** $w = 1$, nothing demands the bond, $\underline{a} = -2$. $A(r)$ increases in $r$
and explodes as $r \to 1 / \beta - 1$, so the bracket $[-0.02, 0.04]$ is safe. The inner tolerances ($10^{-8}$ on
the policy, $10^{-12}$ on the distribution) sit below the outer one ($10^{-6}$ on $r$), since their stopping
errors enter $A(r)$.

In [63]:
huggett = Economy("Huggett", make_grids(-2.0, par, mpar), r -> 1.0, r -> 0.0)
for r in (-0.02, 0.0, 0.02, 0.04)
    @printf("r = %5.1f percent: A(r) = %8.3f\n", 100 * r, aggregate(r, 1.0, huggett.gri).A)
end

r =  -2.0 percent: A(r) =   -1.142
r =   0.0 percent: A(r) =   -0.194
r =   2.0 percent: A(r) =    2.551
r =   4.0 percent: A(r) =   45.937


In [64]:
t0 = time()
rH, aggH, kH = stationary_equilibrium(huggett, -0.02, 0.04, 1e-6)
@printf("Huggett: r* = %.3f percent after %d bisection steps, %.1f s, A(r*) = %.1e\n", 100 * rH, kH, time() - t0, aggH.A)
@printf("share at the limit a = -2: %.3f, and 1/β - 1 = %.2f percent for comparison\n", sum(aggH.lam[1, :]), 100 * (1 / par.β - 1))

Huggett: r* = 0.253 percent after 16 bisection steps, 1.0 s, A(r*) = -6.1e-05
share at the limit a = -2: 0.136, and 1/β - 1 = 4.17 percent for comparison


**Check (slides 18 and 19):** $A(-0.02) \approx -1.1$ and $A(0.04) \approx 46$ bracket the root; $16$ steps
($\log_2(0.06 / 10^{-6}) \approx 16$) to $r^* = 0.25$ percent with $|A(r^*)| < 10^{-4}$, in well under a
second. $r^*$ is far below $1 / \beta - 1 = 4.17$ percent: households save for precaution, and someone has to
hold the savings, so the rate falls until $14$ percent of them sit at the borrowing limit.

**Aiyagari (slides 20 to 22).** Households with $\underline{a} = 0$ and income $w z$, where $z$
is labour efficiency, so aggregate labour is $L = E[z]$. A firm with $Y = K^\alpha L^{1 - \alpha}$,
$\alpha = 0.36$, $\delta = 0.08$, rents capital and labour at competitive prices:

$$
r = \alpha (K / L)^{\alpha - 1} - \delta, \qquad w = (1 - \alpha) (K / L)^\alpha .
$$

The first equation, solved for $K$, is **capital demand** $K^d(r)$, decreasing in $r$. The second gives the wage
that goes with it. The only new step inside the loop: compute $w(r)$ from $K^d(r)$ before solving the household
problem, which the `wage` field of the economy does.

In [65]:
alpha, delta = 0.36, 0.08
L = gri.lam_z' * gri.z                          # aggregate labour: mean efficiency E[z]

# capital demand of the firm: r = alpha (K/L)^(alpha - 1) - delta, solved for K
# K = L ((r + delta) / alpha)^(1 / (alpha - 1)): the formula on session 6, slide 20
K_demand(r) = L * ((r + delta) / alpha)^(1 / (alpha - 1))

# the wage that goes with capital demand at r: w = (1 - alpha) (K/L)^alpha
wage(r) = (1 - alpha) * (K_demand(r) / L)^alpha

wage (generic function with 1 method)

In [66]:
aiyagari = Economy("Aiyagari", make_grids(0.0, par, mpar), wage, K_demand)
r_cm = 1 / par.β - 1                            # the complete markets rate: no precautionary saving, r = 1/β - 1
K_cm = K_demand(r_cm)
@printf("complete markets benchmark: r = %.2f percent, K = %.2f, K/Y = %.2f, w = %.3f\n", 100 * r_cm, K_cm, K_cm / (K_cm^alpha * L^(1 - alpha)), wage(r_cm))

complete markets benchmark: r = 4.17 percent, K = 6.05, K/Y = 2.96, w = 1.178


In [67]:
t0 = time()
rA, aggA, kA = stationary_equilibrium(aiyagari, 0.0, r_cm - 1e-4, 1e-6)
KA = aggA.A
YA = KA^alpha * L^(1 - alpha)
@printf("Aiyagari: r* = %.2f percent after %d steps, %.1f s\n", 100 * rA, kA, time() - t0)
@printf("K* = %.2f (demand %.2f), w* = %.3f, K/Y = %.2f, share at the limit %.3f\n", KA, K_demand(rA), wage(rA), KA / YA, sum(aggA.lam[1, :]))
@printf("against complete markets: K = %.2f, K/Y = %.2f: precautionary saving raises capital by %.0f percent\n", K_cm, K_cm / (K_cm^alpha * L^(1 - alpha)), 100 * (KA / K_cm - 1))

Aiyagari: r* = 2.53 percent after 16 steps, 0.8 s
K* = 7.57 (demand 7.57), w* = 1.278, K/Y = 3.42, share at the limit 0.026
against complete markets: K = 6.05, K/Y = 2.96: precautionary saving raises capital by 25 percent


**Check (slide 21):** $r^* = 2.53$ percent, $K^* = 7.6$, $K / Y = 3.4$, $2.6$ percent of households at
$\underline{a} = 0$. Under complete markets ($r = 1 / \beta - 1 = 4.17$ percent) $K = 6.0$ and $K / Y = 3.0$:
precautionary saving raises the capital stock by a quarter, the point of the paper.

**What the warm starts bought.** Run the Aiyagari bisection once more with cold starts: the same
wrapper, but `aggregate` started from scratch in every step. Compare the time, and the answer.

### 4. Experiment: the settings of the solver, and what they cost

*Session 8, slides 2 and 3.*

Everything above ran with one set of numerical settings: $N_a = 200$ grid points, the policy iterated to
$10^{-8}$, the distribution to $10^{-12}$, the bisection on $r$ to $10^{-6}$. None of these numbers is sacred.
The cell below runs the three pieces of $A(r)$ and the whole Aiyagari equilibrium under five settings, one
change at a time, and prints one line per setting: the time of one $A(r)$, the two iteration counts, the
answer $r^*$ and $K^*$, and the time of the equilibrium.

Read the table with three questions in mind:

1. **The grid.** Doubling $N_a$ doubles the work of every EGM step and every $\mathbf{Q}_{\mathbf{g}}^\top
   \boldsymbol{\lambda}$ product. Does the time double? Do the iteration counts move? Does $r^*$ move, and in
   which digit? (That digit is the grid error of the baseline.)
2. **The tolerances.** Both inner loops converge linearly, so the count grows with the logarithm of the
   tolerance: $10^{-8}$ to $10^{-10}$ should add about as many iterations as $10^{-6}$ to $10^{-8}$. Check.
   Then compare the answers: does $r^*$ move when the policy tolerance is $10^{-6}$, and by how much relative
   to the outer tolerance of $10^{-6}$ on $r$? This is the tolerance hierarchy of the slides: the inner
   stopping errors enter $A(r)$, so they must sit below the outer one.
3. **Where the time goes.** Which setting buys the most time, and what does it cost in digits? The slides say:
   the algorithm sets the order of magnitude, the code the factor. Where in this table is the factor, and
   where would you need another algorithm?

Then change the list yourself. Candidates: $N_z = 5$ or $11$ (the income chain; note that $L = E[z]$ then
changes too, so recompute it), `amax` $= 50$ or $200$ at the same $N_a$ (the spacing near the limit changes),
the bisection tolerance $10^{-4}$ instead of $10^{-6}$, or only the distribution tolerance loosened to
$10^{-8}$ with the policy tolerance kept. Write the setting next to every number you report, as the slides say.


In [68]:
mpar0 = mpar                                    # the baseline settings, restored at the end
settings = [                                    # name, N_a, policy tolerance, distribution tolerance
    ("baseline", 200, 1e-8, 1e-12),
    ("coarser grid, N_a = 100", 100, 1e-8, 1e-12),
    ("finer grid, N_a = 400", 400, 1e-8, 1e-12),
    ("looser tolerances", 200, 1e-6, 1e-8),
    ("tighter tolerances", 200, 1e-10, 1e-14),
]
@printf("%-24s %8s %7s %8s %8s %7s %6s\n", "setting", "A(r) ms", "EGM it", "dist it", "r* (%)", "K*", "eq. s")
for (name, na, crit_pol, crit_dist) in settings
    global mpar = NumericalParameters(; na = na, crit_pol = crit_pol, crit_dist = crit_dist)   # aggregate reads the global mpar
    g = make_grids(0.0, par, mpar)              # the grid of this setting, borrowing limit 0 as in Aiyagari
    (C_, G_, it_), t_hh = best_of(() -> solve_household(r0, w0, par, mpar, g))
    (lam_, it_lam), t_lam = best_of(() -> stationary_dist(transition_matrix(G_, g), mpar, g))
    eco = Economy("Aiyagari", g, wage, K_demand)
    t0 = time()
    r_, agg_, _ = stationary_equilibrium(eco, 0.0, r_cm - 1e-4, 1e-6)
    t_eq = time() - t0
    @printf("%-24s %8.0f %7d %8d %8.3f %7.3f %6.2f\n", name, 1000 * (t_hh + t_lam), it_, it_lam, 100 * r_, agg_.A, t_eq)
end
mpar = mpar0                                    # back to the baseline for anything you run below


setting                   A(r) ms  EGM it  dist it   r* (%)      K*  eq. s
baseline                       82     369     1267    2.534   7.575   1.32
coarser grid, N_a = 100        44     371     1280    2.526   7.584   0.40
finer grid, N_a = 400         171     369     1263    2.537   7.572   2.10
looser tolerances              69     285      723    2.534   7.574   0.45
tighter tolerances             99     450     1539    2.534   7.575   1.01


NumericalParameters(200, 7, 100.0, 1.0e-8, 1.0e-12, 100000)

**What to expect, roughly.** The time of $A(r)$ grows with $N_a$, less than proportionally at these sizes
(the fixed cost of every library call does not grow), while the iteration counts hardly move; $r^*$ changes in the third digit between $N_a = 100$ and $400$, which says how much of the
baseline answer is grid. The tolerances move the counts by a roughly constant number of iterations per factor
$100$, and the answer by less than the outer tolerance when the inner ones sit below it, by more when they do
not. The equilibrium time is the $A(r)$ time times the bisection steps, minus what the warm starts save. If your
numbers say something else, that is the interesting case: find out why.


**At home, if you like.** (i) The wealth distribution at the Aiyagari equilibrium (session 6, slide 22):
the Lorenz curve and the Gini coefficient from `aggA.lam`, mean $7.6$, median $5.3$, the top $10$ percent own
$32$ percent, Gini $0.52$. (ii) Run the Aiyagari bisection once more with `aggregate` started cold in every
step (no `warm` argument) and compare the time: the warm starts save about a quarter of the iterations, no
more, because both inner loops converge linearly. (iii) A third economy for the wrapper: Huggett with a
government that supplies bonds $B = 0.5$, so `demand = lambda r: 0.5`. Where does $r^*$ move, and why?